In [ ]:
# =========================================================
# Imports et configuration
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from config import RANDOM_STATE, TEST_SIZE
from data import load_data, clean_data

sns.set_theme(style="whitegrid")

# =========================================================
# 1. Chargement, Analyse des Proportions et Stratégie
# =========================================================
df = load_data()
df_clean = clean_data(df)
total = len(df_clean)

# Création du dataset purifié de base
df_base = df_clean[
    (df_clean['duration_min'] >= 2.3) & (df_clean['duration_min'] <= 5.0) &
    (df_clean['loudness'] >= -20.0) & (df_clean['tempo'] > 0) & (df_clean['speechiness'] < 0.5)
].copy()

NUMERIC_FEATURES = ['danceability', 'speechiness', 'acousticness', 
                    'instrumentalness', 'liveness', 'valence', 'loudness', 
                    'tempo', 'key', 'mode', 'duration_min', 'release_year']

def get_metrics(name, model, X_te, y_te):
    pred = model.predict(X_te)
    return {
        'Modèle': name,
        'R²': round(r2_score(y_te, pred), 3),
        'MAE': round(mean_absolute_error(y_te, pred), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y_te, pred)), 3)
    }

# =========================================================
# PARTIE A : PRÉDICTION DE L'ÉNERGIE (AVEC ET SANS GENRE)
# =========================================================
print("\n" + "="*50)
print(" PARTIE A : PRÉDICTION DE L'ÉNERGIE (AVEC & SANS GENRE)")
print("="*50)

TARGET_ENR = 'energy'

# 1. Version SANS le genre
X_train_sans, X_test_sans, y_train_sans, y_test_sans = train_test_split(
    df_base[NUMERIC_FEATURES], df_base[TARGET_ENR], test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model_lr_sans = LinearRegression().fit(X_train_sans, y_train_sans)
model_rf_sans = RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=100).fit(X_train_sans, y_train_sans)

# 2. Version AVEC le genre (One-Hot Encoding)
df_encoded = pd.get_dummies(df_base, columns=['playlist_genre'], drop_first=False, dtype=int)
GENRE_FEATURES = [col for col in df_encoded.columns if col.startswith('playlist_genre_')]
FEATURES_ENERGY_AVEC = NUMERIC_FEATURES + GENRE_FEATURES

X_train_avec, X_test_avec, y_train_avec, y_test_avec = train_test_split(
    df_encoded[FEATURES_ENERGY_AVEC], df_encoded[TARGET_ENR], test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model_lr_avec = LinearRegression().fit(X_train_avec, y_train_avec)
model_rf_avec = RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=100).fit(X_train_avec, y_train_avec)

results_nrj = pd.DataFrame([
    get_metrics("Linéaire (Sans Genre)", model_lr_sans, X_test_sans, y_test_sans),
    get_metrics("Linéaire (Avec Genre)", model_lr_avec, X_test_avec, y_test_avec),
    get_metrics("Random Forest (Sans Genre)", model_rf_sans, X_test_sans, y_test_sans),
    get_metrics("Random Forest (Avec Genre)", model_rf_avec, X_test_avec, y_test_avec),
])
display(results_nrj)

# =========================================================
# PARTIE B : PRÉDICTION DE LA POPULARITÉ (SANS GENRE - CONTRAINTE EXERCICE)
# =========================================================
print("\n" + "="*50)
print(" PARTIE B : PRÉDICTION DE LA POPULARITÉ")
print("="*50)

TARGET_POP = 'track_popularity'

X_train_pop, X_test_pop, y_train_pop, y_test_pop = train_test_split(
    df_base[NUMERIC_FEATURES], df_base[TARGET_POP], test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model_pop_lr = LinearRegression().fit(X_train_pop, y_train_pop)
model_pop_rf = RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=100, max_depth=15).fit(X_train_pop, y_train_pop)

results_pop = pd.DataFrame([
    get_metrics("Linéaire (Popularité)", model_pop_lr, X_test_pop, y_test_pop),
    get_metrics("Random Forest (Popularité)", model_pop_rf, X_test_pop, y_test_pop)
])
display(results_pop)

=== ANALYSE STATISTIQUE DES VALEURS ABERRANTES (OUTLIERS) ===
Taille du jeu de données brut : 28068 morceaux



,Critère aberrant,Nombre de pistes,Proportion (%),Stratégie
0,Durée < 2.3m ou > 5.0m,3898,13.89,Suppression
1,Loudness < -20 dB,62,0.22,Suppression
2,Tempo = 0 BPM,1,0.00,Suppression
3,Speechiness >= 0.5,139,0.50,Suppression



=== DÉBAT : SUPPRESSION VS REMPLACEMENT ===
Conclusion : Ces anomalies ou choix stricts (format radio 2.3-5 min) représentant une proportion maîtrisée du volume global, la suppression définitive est la méthode la plus rigoureuse.


 PARTIE A : PRÉDICTION DE L'ÉNERGIE


,Modèle,R²,MAE,RMSE
0,Linéaire Simple (Loudness),0.452,0.105,0.132
1,Linéaire Simple (Acousticness),0.296,0.122,0.150
2,Linéaire Multiple (+ Genres),0.647,0.083,0.106
3,RF Simple (Loudness),0.277,0.119,0.152
4,RF Simple (Acousticness),0.222,0.126,0.157
5,RF Multiple (+ Genres),0.719,0.073,0.094



 PARTIE B : PRÉDICTION DE LA POPULARITÉ (BONUS)


,Modèle,R²,MAE,RMSE
0,Linéaire Simple (Loudness),0.002,19.694,23.543
1,Linéaire Simple (Danceability),0.002,19.700,23.541
2,Linéaire Multiple (+ Genres),0.085,18.783,22.538
3,RF Multiple (+ Genres),0.170,17.705,21.474



=== RÉPONSE À LA QUESTION BONUS ===
La précision s'effondre. La popularité ne dépend pas que des attributs audio et du genre, mais d'éléments externes (marketing, notoriété de l'artiste, tendances TikTok, etc.) que la machine ne possède pas.

Listes de variables sauvegardées avec succès.
